In [1]:
import re
import pandas as pd
from pathlib   import Path
from Config_utils import cfg, SPECIAL
from collections import Counter
from typing import List, Dict, Tuple

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report



In [2]:
%run '0 - DF Functions.ipynb' -i

In [3]:
current_dir = Path.cwd() 
file_path = current_dir.parent / "data" / cfg.file_name
df = pd.read_csv(file_path)

Converting product group into indexes

In [4]:
product_groups = df['Product_Group'].unique()

d_products ={product: i for i, product in enumerate(product_groups)}

d_products

df['Product_Group_ID'] = df['Product_Group'].map(d_products)


Introducing frustration and Regulatory indices

In [5]:
# 1. Define High-Frustration Keywords
frustration_words = ['angry', 'terrible', 'rude', 'fix', 'waited', 'stole', 'illegal']

# Apply logic: 1 if any word exists in text, else 0
df['Frustration_Index'] = df['Consumer complaint narrative'].str.lower().apply(
    lambda x: 1 if any(word in x for word in frustration_words) else 0)


In [6]:
# 2. Define Regulatory/Legal Keywords
regulatory_words = ['cfpb', 'lawyer', 'legal', 'fraud', 'report', 'sue', 'illegal']

df['Regulatory_Flag'] = df['Consumer complaint narrative'].str.lower().apply(
    lambda x: 1 if any(word in x for word in regulatory_words) else 0
)

In [7]:
def clean_text(text):
    text = text.lower()
    #removing musked strings XXXX
    # text = re.sub(r'x+', '', text)
    #removing punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # text = re.sub(r'\d+', '', text)

    lemmatizer = WordNetLemmatizer()
    # stop_words = set(stopwords.words('english'))

    words = text.split()
    cleaned_words = [lemmatizer.lemmatize(w) for w in words] # if w not in stop_words]

    return cleaned_words


In [8]:
df['tokens'] = df['Consumer complaint narrative'].apply(clean_text)

### Split train/val/test

In [9]:
n = len(df)
n_train = int(cfg.train_ratio * n)
n_val = int(cfg.val_ratio * n)
n_test = n - n_train - n_val


train_complains = df[:n_train].copy()
val_complains = df[n_train:n_train+n_val].copy()
test_complains = df[n_train+n_val:].copy()

def num_tokens(slist: List[List[str]]) -> int:
    return sum(len(s) for s in slist)


print("Train:", f"{len(train_complains):,}", "complains |", f"{num_tokens(train_complains['tokens']):,}", "tokens")
print("Val  :", f"{len(val_complains):,}",   "complains |", f"{num_tokens(val_complains['tokens']):,}",   "tokens")
print("Test :", f"{len(test_complains):,}",  "complains |", f"{num_tokens(test_complains['tokens']):,}",  "tokens")


Train: 32,000 complains | 6,877,992 tokens
Val  : 4,000 complains | 841,129 tokens
Test : 4,000 complains | 665,360 tokens


### Vocabulary generation (train only)

In [10]:
all_tokens = [ token for tokens in train_complains['tokens'] for token in tokens]

all_tokens_counters = Counter(all_tokens)

all_tokens_counters.most_common(20)

[('xxxx', 342237),
 ('the', 291606),
 ('to', 236694),
 ('i', 231802),
 ('and', 193904),
 ('a', 172429),
 ('my', 141427),
 ('of', 120347),
 ('that', 100592),
 ('wa', 87325),
 ('in', 75702),
 ('on', 74567),
 ('this', 69960),
 ('they', 67820),
 ('account', 67040),
 ('for', 65089),
 ('not', 65024),
 ('have', 60446),
 ('me', 58904),
 ('is', 58604)]

In [11]:
vocab = {}

for i, (key, item) in enumerate(SPECIAL.items()):
    vocab[key] = i

for word, count in all_tokens_counters.items():
    if count >= cfg.min_freq:
        vocab[word] = len(vocab)


vocab_size = len(vocab)
UNK_pos = vocab.get('UNK')
START_pos = vocab.get('BOS')
END_pos = vocab.get('EOS')

print("min_freq:", cfg.min_freq)
print(f"vocab_size:  {vocab_size:,}")
print("UNK id:", UNK_pos)

min_freq: 3
vocab_size:  15,723
UNK id: 2


In [12]:
def text_to_indices (tokens, vocab):
    indices = [vocab.get(token, UNK_pos) for token in tokens]

    return indices
     

In [13]:
train_complains['Complain_coded'] = train_complains['tokens'].apply(text_to_indices, vocab=vocab)
val_complains['Complain_coded'] = val_complains['tokens'].apply(text_to_indices, vocab=vocab)
test_complains['Complain_coded'] = test_complains['tokens'].apply(text_to_indices, vocab=vocab)



In [14]:
get_dataframe_stats_per_column(train_complains)


Column [Unnamed: 0] has 29,617 unique values, max/min content lenghts 8/3 ; avg lenght 7

Printing top 10 records out of 29,617
-------------------------------------------------------------
Value                | Total Records        | Percentage     
-------------------------------------------------------------
1123112              | 7                    | 0.02%
2313308              | 6                    | 0.02%
7135610              | 6                    | 0.02%
10243558             | 6                    | 0.02%
4212488              | 6                    | 0.02%
2341364              | 6                    | 0.02%
7031444              | 6                    | 0.02%
474727               | 6                    | 0.02%
13260331             | 6                    | 0.02%
3637345              | 6                    | 0.02%
10922730             | 6                    | 0.02%

Column [Date received] has 3,851 unique values, max/min content lenghts 10/10 ; avg lenght 10

Printing top 10 r